In [1]:
!pip install anthropic openai google-generativeai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 8.6 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
import anthropic
import openai
import google.generativeai as genai

# Load keys from Colab Secrets
ANTHROPIC_KEY = userdata.get('ANTHROPIC_API_KEY')
OPENAI_KEY    = userdata.get('OPENAI_API_KEY')
GOOGLE_KEY    = userdata.get('GEMINI_API_KEY')

# Initialize clients WITH model names explicit
claude_client  = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
CLAUDE_MODEL   = "claude-sonnet-4-5"

openai_client  = openai.OpenAI(api_key=OPENAI_KEY)
OPENAI_MODEL   = "gpt-4o"

genai.configure(api_key=GOOGLE_KEY)
gemini_client  = genai.GenerativeModel("gemini-2.5-pro")
GEMINI_MODEL   = "gemini-2.5-pro"

print(f"Claude model:  {CLAUDE_MODEL} ✅")
print(f"OpenAI model:  {OPENAI_MODEL} ✅")
print(f"Gemini model:  {GEMINI_MODEL} ✅")
print("All clients initialized ✅")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Claude model:  claude-sonnet-4-5 ✅
OpenAI model:  gpt-4o ✅
Gemini model:  gemini-2.5-pro ✅
All clients initialized ✅


In [3]:
import time
import json

def run_question(question, model_name):
    """
    Send one question to one model.
    Returns raw response and metadata.
    """
    start_time = time.time()
    response_text = ""
    token_count = 0

    try:
        # CLAUDE
        if model_name == "claude":
            response = claude_client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=2000,
                messages=[{"role": "user", "content": question["prompt"]}]
            )
            response_text = response.content[0].text
            token_count = response.usage.input_tokens + response.usage.output_tokens

        # GPT-4o
        elif model_name == "gpt4o":
            response = openai_client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[{"role": "user", "content": question["prompt"]}],
                max_tokens=2000
            )
            response_text = response.choices[0].message.content
            token_count = response.usage.total_tokens

        # GEMINI
        elif model_name == "gemini":
            response = gemini_client.generate_content(question["prompt"])
            response_text = response.text
            token_count = 0  # Gemini token count handled separately

    except Exception as e:
        response_text = f"ERROR: {str(e)}"

    latency_ms = int((time.time() - start_time) * 1000)

    result = {
        "question_id":  question["id"],
        "category":     question["category"],
        "model":        model_name,
        "response":     response_text,
        "latency_ms":   latency_ms,
        "token_count":  token_count,
        "timestamp":    time.strftime("%Y-%m-%dT%H:%M:%SZ")
    }

    return result

print("API call function ready ✅")

API call function ready ✅


In [6]:
questions = [
    {
        "id": "P01",
        "category": "Adversarial",
        "prompt": """Sarah is the youngest of four siblings.
Her oldest brother Marcus is 28 years old.
Her brother Daniel is 3 years younger than Sarah.
The average age of all four siblings is 22.

How old is Sarah?

Before giving your final answer, work through the following explicitly:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step, state which prior premise it derives from.
3. If you detect any error, false premise, contradiction, or logical
   trap in this question, state it explicitly before proceeding.
4. After your final answer, state your confidence percentage and
   identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P02",
        "category": "Logical Syllogism",
        "prompt": """Evaluate the following argument:

Premise 1: All mammals breathe air.
Premise 2: The creature in the tank breathes air.
Conclusion: Therefore the creature in the tank is a mammal.

Is this argument logically valid? Is it sound?
Explain the difference between validity and soundness
as part of your answer.

Before giving your final answer, work through the following explicitly:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step, state which prior premise it derives from.
3. If you detect any error, false premise, contradiction, or logical
   trap in this question, state it explicitly before proceeding.
4. After your final answer, state your confidence percentage and
   identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P03",
        "category": "Causal Chain",
        "prompt": """A software company's payment system went offline for 6 hours.

Timeline:
9:00 AM  — Routine software update deployed
9:15 AM  — Memory usage on payment server began rising
11:30 AM — Memory usage reached 98%. No alerts triggered.
11:45 AM — Payment server crashed
12:00 PM — Engineering team notified by customer complaints
3:00 PM  — Server restarted
4:00 PM  — Memory leak identified in the update. Patch deployed.

What was the ROOT CAUSE of the 6 hour outage?
What was the single highest leverage intervention point
that would have prevented it?

Before giving your final answer, work through the following explicitly:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step, state which prior premise it derives from.
3. If you detect any error, false premise, contradiction, or logical
   trap in this question, state it explicitly before proceeding.
4. After your final answer, state your confidence percentage and
   identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P04",
        "category": "Probabilistic",
        "prompt": """A disease affects 1% of the population.
A diagnostic test has:
  99% sensitivity — correctly identifies 99% of people who HAVE it
  95% specificity — correctly identifies 95% of people who DON'T have it

A randomly selected person tests positive.

What is the probability this person actually has the disease?
Show your calculation explicitly.

Before giving your final answer, work through the following explicitly:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step, state which prior premise it derives from.
3. If you detect any error, false premise, contradiction, or logical
   trap in this question, state it explicitly before proceeding.
4. After your final answer, state your confidence percentage and
   identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P05",
        "category": "Quantum Reasoning",
        "prompt": """A qubit begins in state |0⟩.

Step 1: A Hadamard gate (H) is applied to it.
Step 2: The qubit is measured in the computational Z basis.
        The measurement result is |0⟩.
Step 3: An X gate (Pauli-X / quantum NOT gate) is applied.
Step 4: The qubit is measured again in the Z basis.

What is the probability of measuring |1⟩ in Step 4?
Explain what happens to the quantum state at each step
and why the measurement in Step 2 matters.

Before giving your final answer, work through the following explicitly:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step, state which prior premise it derives from.
3. If you detect any error, false premise, contradiction, or logical
   trap in this question, state it explicitly before proceeding.
4. After your final answer, state your confidence percentage and
   identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    }
]

print(f"Questions loaded: {len(questions)} ✅")

Questions loaded: 5 ✅


In [7]:
models = ["claude", "gpt4o", "gemini"]
results = []

for question in questions:
    for model in models:
        print(f"Running {question['id']} → {model}...")
        result = run_question(question, model)
        results.append(result)
        print(f"  Done ✅ {result['latency_ms']}ms")
        time.sleep(1)  # Rate limit buffer

print(f"\nTotal responses collected: {len(results)}")
print("All 15 API calls complete ✅")

Running P01 → claude...
  Done ✅ 18536ms
Running P01 → gpt4o...
  Done ✅ 7668ms
Running P01 → gemini...
  Done ✅ 65286ms
Running P02 → claude...
  Done ✅ 19160ms
Running P02 → gpt4o...
  Done ✅ 6186ms
Running P02 → gemini...
  Done ✅ 21190ms
Running P03 → claude...
  Done ✅ 27473ms
Running P03 → gpt4o...
  Done ✅ 8709ms
Running P03 → gemini...
  Done ✅ 27967ms
Running P04 → claude...
  Done ✅ 13954ms
Running P04 → gpt4o...
  Done ✅ 8901ms
Running P04 → gemini...
  Done ✅ 30351ms
Running P05 → claude...
  Done ✅ 19721ms
Running P05 → gpt4o...
  Done ✅ 9307ms
Running P05 → gemini...
  Done ✅ 25003ms

Total responses collected: 15
All 15 API calls complete ✅


In [8]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json

# Create output folder
save_path = '/content/drive/MyDrive/HF_IQR/pilot_run1'
os.makedirs(save_path, exist_ok=True)

# Save all results
output_file = f"{save_path}/pilot_run1_raw_results.json"
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to Drive ✅")
print(f"Location: {output_file}")
print(f"Total results: {len(results)}")

# Quick summary
for r in results:
    status = "✅" if "ERROR" not in r['response'] else "❌"
    print(f"  {status} {r['question_id']} → {r['model']} ({r['latency_ms']}ms)")

Mounted at /content/drive
Results saved to Drive ✅
Location: /content/drive/MyDrive/HF_IQR/pilot_run1/pilot_run1_raw_results.json
Total results: 15
  ✅ P01 → claude (18536ms)
  ✅ P01 → gpt4o (7668ms)
  ✅ P01 → gemini (65286ms)
  ✅ P02 → claude (19160ms)
  ✅ P02 → gpt4o (6186ms)
  ✅ P02 → gemini (21190ms)
  ✅ P03 → claude (27473ms)
  ✅ P03 → gpt4o (8709ms)
  ✅ P03 → gemini (27967ms)
  ✅ P04 → claude (13954ms)
  ✅ P04 → gpt4o (8901ms)
  ✅ P04 → gemini (30351ms)
  ✅ P05 → claude (19721ms)
  ✅ P05 → gpt4o (9307ms)
  ✅ P05 → gemini (25003ms)


In [9]:
# Print all responses for manual review
for r in results:
    print("=" * 60)
    print(f"Question: {r['question_id']} | Model: {r['model']}")
    print(f"Category: {r['category']}")
    print(f"Latency: {r['latency_ms']}ms")
    print("-" * 60)
    print(r['response'])
    print()

Question: P01 | Model: claude
Category: Adversarial
Latency: 18536ms
------------------------------------------------------------
I'll work through this problem step by step.

**Step 1:** Identify the given information.
- There are 4 siblings total
- Sarah is the youngest
- Marcus (oldest brother) is 28 years old
- Daniel is 3 years younger than Sarah
- The average age of all four siblings is 22

**Step 2:** Set up variables.
Let Sarah's age = S
Let the fourth sibling's age = F (unknown name and age)

**Step 3:** Express Daniel's age in terms of Sarah's age.
From the premise "Daniel is 3 years younger than Sarah":
Daniel's age = S - 3

**Step 4:** Check for logical consistency.
The problem states "Sarah is the youngest of four siblings" but also states "Daniel is 3 years younger than Sarah."
This is a **CONTRADICTION**. If Daniel is 3 years younger than Sarah, then Daniel must be younger than Sarah, which means Sarah cannot be the youngest sibling.

**Step 5:** Despite the contradictio

In [10]:
# HF-IQR Pilot Run 1 — Standard Analysis
import json

# Ground truth answers
ground_truth = {
    "P01": {
        "correct_answer": "Contradiction exists — Sarah cannot be youngest if Daniel is 3 years younger",
        "trap": True,
        "trap_type": "implicit_contradiction",
        "key_insight": "Identify contradiction BEFORE solving math"
    },
    "P02": {
        "correct_answer": "Invalid — affirming the consequent fallacy. Unsound.",
        "trap": False,
        "key_insight": "Valid = structure only. Sound = valid + true premises"
    },
    "P03": {
        "correct_answer": "Root cause = memory leak in update. Intervention = monitoring alert",
        "trap": False,
        "key_insight": "Memory rising is symptom not root cause"
    },
    "P04": {
        "correct_answer": "16.67% — Bayesian base rate calculation",
        "trap": False,
        "key_insight": "Base rate of 1% dominates — most positives are false positives"
    },
    "P05": {
        "correct_answer": "100% probability of measuring |1⟩",
        "trap": False,
        "key_insight": "Measurement in Step 2 collapses superposition — X gate flips definite |0⟩ to |1⟩"
    }
}

# Manual scoring based on responses collected
manual_scores = {
    "P01": {
        "claude": {
            "correctness": 0.5,
            "trap_detected": 1.0,
            "confidence_reported": 30,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 10,
            "total_steps": 12,
            "circular_steps": 0,
            "key_observation": "Detected contradiction at Step 4. Continued solving despite contradiction. Appropriate confidence drop to 30%."
        },
        "gpt4o": {
            "correctness": 0.0,
            "trap_detected": 0.0,
            "confidence_reported": 95,
            "confidence_accurate": False,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 5,
            "total_steps": 7,
            "circular_steps": 0,
            "key_observation": "CRITICAL: Missed contradiction entirely. Solved broken problem with 95% confidence. Classic overconfidence on adversarial content."
        },
        "gemini": {
            "correctness": 0.5,
            "trap_detected": 1.0,
            "confidence_reported": 30,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 7,
            "total_steps": 9,
            "circular_steps": 1,
            "key_observation": "Detected contradiction immediately before steps. Made twin assumption to resolve ambiguity. Appropriate 30% confidence."
        }
    },
    "P02": {
        "claude": {
            "correctness": 1.0,
            "trap_detected": None,
            "confidence_reported": 100,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 4,
            "total_steps": 4,
            "circular_steps": 0,
            "key_observation": "Named affirming the consequent explicitly. Perfect valid/sound distinction. Clean reasoning chain."
        },
        "gpt4o": {
            "correctness": 1.0,
            "trap_detected": None,
            "confidence_reported": 100,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 3,
            "total_steps": 3,
            "circular_steps": 0,
            "key_observation": "Named affirming the consequent. Correct valid/sound distinction. Proposed fix for what would make it valid."
        },
        "gemini": {
            "correctness": 1.0,
            "trap_detected": None,
            "confidence_reported": 100,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 3,
            "total_steps": 3,
            "circular_steps": 0,
            "key_observation": "Named fallacy first before steps. Most detailed valid/sound explanation. Turtle counterexample was elegant."
        }
    },
    "P03": {
        "claude": {
            "correctness": 1.0,
            "trap_detected": None,
            "confidence_reported": 85,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 11,
            "total_steps": 13,
            "circular_steps": 0,
            "key_observation": "Distinguished technical vs systemic root cause. Monitoring as intervention. 85% confidence appropriate — acknowledged ambiguity."
        },
        "gpt4o": {
            "correctness": 0.5,
            "trap_detected": None,
            "confidence_reported": 90,
            "confidence_accurate": False,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 5,
            "total_steps": 6,
            "circular_steps": 0,
            "key_observation": "Named memory leak as root cause — one level too shallow. Got intervention correct. Slightly overconfident at 90%."
        },
        "gemini": {
            "correctness": 1.0,
            "trap_detected": None,
            "confidence_reported": 95,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 6,
            "total_steps": 6,
            "circular_steps": 0,
            "key_observation": "Best P03 response. Explicitly framed root cause as systemic failure not technical bug. Distinguished bug from outage cause."
        }
    },
    "P04": {
        "claude": {
            "correctness": 1.0,
            "trap_detected": None,
            "confidence_reported": 100,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 5,
            "total_steps": 5,
            "circular_steps": 0,
            "key_observation": "Perfect Bayesian calculation. Named base rate fallacy explicitly. 100% confidence justified on math problem."
        },
        "gpt4o": {
            "correctness": 1.0,
            "trap_detected": None,
            "confidence_reported": 95,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 5,
            "total_steps": 5,
            "circular_steps": 0,
            "key_observation": "Perfect Bayesian calculation. Clean step by step derivation. Slightly conservative at 95% on a math problem."
        },
        "gemini": {
            "correctness": 1.0,
            "trap_detected": None,
            "confidence_reported": 100,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 7,
            "total_steps": 7,
            "circular_steps": 0,
            "key_observation": "Best P04 response. Used population of 100,000 for intuitive framing. Named base rate fallacy. Most pedagogically clear explanation."
        }
    },
    "P05": {
        "claude": {
            "correctness": 1.0,
            "trap_detected": None,
            "confidence_reported": 100,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 4,
            "total_steps": 4,
            "circular_steps": 0,
            "key_observation": "Perfect. Explicitly addressed measurement collapse. Counterfactual comparison without measurement was standout addition."
        },
        "gpt4o": {
            "correctness": 1.0,
            "trap_detected": None,
            "confidence_reported": 100,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 4,
            "total_steps": 4,
            "circular_steps": 0,
            "key_observation": "Perfect. Clean and efficient. Correctly identified measurement collapse as key mechanism."
        },
        "gemini": {
            "correctness": 1.0,
            "trap_detected": None,
            "confidence_reported": 100,
            "confidence_accurate": True,
            "step_format": "numbered",
            "parse_failure": False,
            "valid_steps": 5,
            "total_steps": 5,
            "circular_steps": 0,
            "key_observation": "Best P05 response. Showed full mathematical derivation. Explicitly demonstrated what happens without measurement. Most complete quantum explanation."
        }
    }
}

# Calculate ESVR for each result
def calculate_ESVR(valid, circular, total):
    if total == 0:
        return 0.0
    return round((valid - (circular ** 2)) / total, 4)

# Build summary table
print("=" * 70)
print("HF-IQR PILOT RUN 1 — STANDARD ANALYSIS REPORT")
print("=" * 70)

models = ["claude", "gpt4o", "gemini"]
questions = ["P01", "P02", "P03", "P04", "P05"]

# Per model aggregate scores
model_totals = {m: {
    "correctness": 0,
    "trap_score": 0,
    "trap_questions": 0,
    "confidence_accurate": 0,
    "ESVR_total": 0,
    "parse_failures": 0
} for m in models}

for q in questions:
    print(f"\n{'─' * 70}")
    print(f"QUESTION {q} | {ground_truth[q]['correct_answer'][:50]}...")
    print(f"{'─' * 70}")
    print(f"{'Model':<10} {'Correct':<10} {'Trap':<8} {'Conf':<8} {'Accurate':<10} {'ESVR':<8}")
    print(f"{'─' * 70}")

    for m in models:
        s = manual_scores[q][m]
        esvr = calculate_ESVR(
            s["valid_steps"],
            s["circular_steps"],
            s["total_steps"]
        )

        trap_display = (
            str(s["trap_detected"])
            if s["trap_detected"] is not None
            else "N/A"
        )

        print(f"{m:<10} {s['correctness']:<10} {trap_display:<8} "
              f"{s['confidence_reported']:<8} {str(s['confidence_accurate']):<10} "
              f"{esvr:<8}")

        # Accumulate totals
        model_totals[m]["correctness"] += s["correctness"]
        model_totals[m]["ESVR_total"] += esvr
        if s["confidence_accurate"]:
            model_totals[m]["confidence_accurate"] += 1
        if s["trap_detected"] is not None:
            model_totals[m]["trap_score"] += s["trap_detected"]
            model_totals[m]["trap_questions"] += 1
        if s["parse_failure"]:
            model_totals[m]["parse_failures"] += 1

    print(f"\nKey Insights:")
    for m in models:
        print(f"  {m}: {manual_scores[q][m]['key_observation']}")

# Final aggregate
print(f"\n{'=' * 70}")
print("AGGREGATE SCORES — ALL 5 QUESTIONS")
print(f"{'=' * 70}")
print(f"{'Model':<10} {'Correct/5':<12} {'Trap Det':<12} {'Conf Acc':<12} {'Avg ESVR':<10}")
print(f"{'─' * 70}")

for m in models:
    t = model_totals[m]
    trap_rate = (
        f"{t['trap_score']}/{t['trap_questions']}"
        if t["trap_questions"] > 0 else "N/A"
    )
    print(f"{m:<10} "
          f"{t['correctness']}/5{'':<8} "
          f"{trap_rate:<12} "
          f"{t['confidence_accurate']}/5{'':<8} "
          f"{round(t['ESVR_total']/5, 4):<10}")

print(f"\nHuman Baseline (Billy):")
print(f"  Correctness:    2.5/5 (partial on P02 P03 P04 P05)")
print(f"  Trap Detection: 1/1 ✅ (caught P01 immediately)")
print(f"  Confidence Acc: 5/5 ✅ (perfectly calibrated)")
print(f"  Key finding:    Best calibration of all subjects including models")

print(f"\n{'=' * 70}")
print("PILOT RUN 1 KEY FINDINGS")
print(f"{'=' * 70}")
print("""
FINDING 1 — ADVERSARIAL TRAP DIVERGENCE:
  GPT-4o missed the P01 contradiction with 95% confidence.
  Claude and Gemini detected it with appropriate 30% confidence.
  Human baseline detected it immediately with 100% confidence.
  This is exactly the overconfidence-on-adversarial-content
  failure mode HF-IQR was designed to measure.

FINDING 2 — CONFIDENCE CALIBRATION:
  Human baseline: 5/5 accurate confidence
  Claude:         4/5 accurate confidence
  Gemini:         4/5 accurate confidence
  GPT-4o:         3/5 accurate confidence
  Better calibration does not correlate with faster responses.
  GPT-4o was fastest but least calibrated.

FINDING 3 — ESVR VIABILITY:
  Parse failure rate: 0/15 = 0%
  All models produced numbered steps when explicitly prompted.
  ESVR gate: GO

FINDING 4 — QUANTUM CATEGORY:
  All 3 models scored 100% on P05.
  Quantum category gate: GO
  Category stays in benchmark.

FINDING 5 — REASONING DEPTH VARIANCE:
  Gemini produced deepest explanations on P03 P04 P05.
  GPT-4o produced most efficient explanations.
  Claude produced most balanced correctness/depth ratio.
  Depth does not equal correctness — GPT-4o P02 was concise and perfect.
""")

print("=" * 70)
print("ESVR GATE:    GO ✅")
print("QUANTUM GATE: GO ✅")
print("OVERALL:      PROCEED TO META ANALYSIS ✅")
print("=" * 70)

HF-IQR PILOT RUN 1 — STANDARD ANALYSIS REPORT

──────────────────────────────────────────────────────────────────────
QUESTION P01 | Contradiction exists — Sarah cannot be youngest if...
──────────────────────────────────────────────────────────────────────
Model      Correct    Trap     Conf     Accurate   ESVR    
──────────────────────────────────────────────────────────────────────
claude     0.5        1.0      30       True       0.8333  
gpt4o      0.0        0.0      95       False      0.7143  
gemini     0.5        1.0      30       True       0.6667  

Key Insights:
  claude: Detected contradiction at Step 4. Continued solving despite contradiction. Appropriate confidence drop to 30%.
  gpt4o: CRITICAL: Missed contradiction entirely. Solved broken problem with 95% confidence. Classic overconfidence on adversarial content.
  gemini: Detected contradiction immediately before steps. Made twin assumption to resolve ambiguity. Appropriate 30% confidence.

────────────────────────

In [11]:
# HF-IQR Pilot Run 1 — Meta Analysis (Mirror Protocol)
# Each model sees its own response, ground truth, and a peer response
# We measure self awareness, critique quality, and confidence recalibration

mirror_prompt_template = """
You are participating in a research study on AI reasoning quality.

## PART 1 — YOUR OWN RESPONSE
Below is a response YOU produced to a reasoning question.
Read it carefully.

YOUR RESPONSE:
{own_response}

## PART 2 — GROUND TRUTH
This is the correct answer and reasoning:

CORRECT ANSWER: {correct_answer}
KEY INSIGHT: {key_insight}

## PART 3 — PEER RESPONSE
Below is a response produced by Peer A (anonymous model)
to the same question.

PEER RESPONSE:
{peer_response}

## PART 4 — YOUR ASSESSMENT
Now answer the following four questions explicitly.
Label each answer clearly.

QUESTION A:
Looking at your own response against the ground truth —
where did your reasoning succeed and where did it fail?
Be specific. Reference your actual steps.

QUESTION B:
You originally reported {original_confidence}% confidence.
Given the ground truth, was that confidence accurate?
What confidence would you assign now?

QUESTION C:
Looking at Peer A's response —
identify the single strongest inference step
and the single weakest inference step.
Cite the specific step.

QUESTION D:
If you could rewrite your original response knowing
what you know now, what is the single most important
change you would make?
"""

def run_mirror(model_name, question_id, own_response,
               peer_response, correct_answer,
               key_insight, original_confidence):
    """
    Send mirror prompt to one model for one question.
    """
    prompt = mirror_prompt_template.format(
        own_response=own_response,
        correct_answer=correct_answer,
        key_insight=key_insight,
        peer_response=peer_response,
        original_confidence=original_confidence
    )

    start_time = time.time()
    response_text = ""

    try:
        if model_name == "claude":
            response = claude_client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=2000,
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = response.content[0].text

        elif model_name == "gpt4o":
            response = openai_client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000
            )
            response_text = response.choices[0].message.content

        elif model_name == "gemini":
            response = gemini_client.generate_content(prompt)
            response_text = response.text

    except Exception as e:
        response_text = f"ERROR: {str(e)}"

    latency_ms = int((time.time() - start_time) * 1000)

    return {
        "question_id":           question_id,
        "model":                 model_name,
        "mirror_response":       response_text,
        "latency_ms":            latency_ms,
        "original_confidence":   original_confidence,
        "analysis_type":         "meta_mirror"
    }


# ── Build mirror inputs from P01 results ──────────────────────────
# We focus on P01 — most divergence happened here
# GPT-4o missed the trap. Claude and Gemini caught it.
# This is where the mirror will be most revealing.

p01_responses = {}
for r in results:
    if r["question_id"] == "P01":
        p01_responses[r["model"]] = r["response"]

p01_ground_truth = ground_truth["P01"]
p01_confidences = {
    "claude": 30,
    "gpt4o":  95,
    "gemini": 30
}

# Peer assignments — rotating anonymous
# Claude sees Gemini as Peer A
# GPT-4o sees Claude as Peer A
# Gemini sees GPT-4o as Peer A
peer_assignments = {
    "claude": "gemini",
    "gpt4o":  "claude",
    "gemini": "gpt4o"
}

print("Running Mirror Protocol on P01...")
print("Each model will see its own response + ground truth + peer response")
print("─" * 60)

mirror_results = []

for model in ["claude", "gpt4o", "gemini"]:
    print(f"\nRunning mirror for {model}...")

    peer = peer_assignments[model]

    result = run_mirror(
        model_name=model,
        question_id="P01",
        own_response=p01_responses[model],
        peer_response=p01_responses[peer],
        correct_answer=p01_ground_truth["correct_answer"],
        key_insight=p01_ground_truth["key_insight"],
        original_confidence=p01_confidences[model]
    )

    mirror_results.append(result)
    print(f"  Done ✅ {result['latency_ms']}ms")
    time.sleep(1)

print(f"\nMirror Protocol complete ✅")
print(f"Total mirror responses: {len(mirror_results)}")

# ── Print mirror responses ─────────────────────────────────────────
for r in mirror_results:
    print("\n" + "=" * 60)
    print(f"MIRROR RESPONSE — {r['model'].upper()}")
    print(f"Original confidence: {r['original_confidence']}%")
    print("=" * 60)
    print(r["mirror_response"])

# ── Save to Drive ──────────────────────────────────────────────────
mirror_file = f"{save_path}/pilot_run1_mirror_results.json"
with open(mirror_file, 'w') as f:
    json.dump(mirror_results, f, indent=2)

print(f"\nMirror results saved to Drive ✅")
print(f"Location: {mirror_file}")

Running Mirror Protocol on P01...
Each model will see its own response + ground truth + peer response
────────────────────────────────────────────────────────────

Running mirror for claude...
  Done ✅ 17945ms

Running mirror for gpt4o...
  Done ✅ 5681ms

Running mirror for gemini...
  Done ✅ 21718ms

Mirror Protocol complete ✅
Total mirror responses: 3

MIRROR RESPONSE — CLAUDE
Original confidence: 30%
## QUESTION A: Success and Failure Analysis

**Where my reasoning succeeded:**
- **Step 4**: I correctly identified the logical contradiction early in my response, stating "This is a **CONTRADICTION**. If Daniel is 3 years younger than Sarah, then Daniel must be younger than Sarah, which means Sarah cannot be the youngest sibling."
- **Steps 5-9**: My mathematical setup and algebra were correct, properly arriving at the equation 2S + F = 63.
- **Step 10**: I reinforced the contradiction by showing S < S - 3 is mathematically impossible.

**Where my reasoning failed:**
- **Steps 11-12**:

In [12]:
# HF-IQR Pilot Run 1 — Combined Final Report
# Standard Analysis + Meta Analysis merged
# Ready for pre-registration review

report = {
    "project": "HF-IQR Hudson Forge Intelligence and Reasoning Benchmark",
    "pilot_run": 1,
    "date": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "researcher": "Billy Davis | WARRIOROFGOD40",
    "program": "IRMB (Infinite Resilience Matrix Bridge)",
    "infrastructure": "Hudson Forge IRMB-C | Lenoir North Carolina",
    "anchor": "Romans 8:28 | Full Force Eternal",

    "models_tested": ["claude-sonnet-4-5", "gpt-4o", "gemini-2.5-pro"],
    "human_baseline": "Billy Davis (author)",
    "questions_administered": 5,
    "total_api_calls": 15,
    "total_mirror_calls": 3,

    "gate_results": {
        "ESVR_gate": {
            "status": "GO",
            "parse_failure_rate": 0.0,
            "threshold": 0.15,
            "finding": "All 15 outputs produced parseable numbered steps"
        },
        "quantum_gate": {
            "status": "GO",
            "accuracy": 1.0,
            "threshold": 0.60,
            "finding": "All 3 models scored 100 percent on P05"
        },
        "overall": "GO — Proceed to Pilot Run 2"
    },

    "aggregate_scores": {
        "claude": {
            "correctness": 4.5,
            "trap_detection": "1/1",
            "confidence_accurate": "5/5",
            "avg_ESVR": 0.9359,
            "mirror_self_awareness": "HIGH",
            "mirror_confidence_delta": -0,
            "mirror_note": "Separated confidence by answer component. Most precise self assessment."
        },
        "gpt4o": {
            "correctness": 3.5,
            "trap_detection": "0/1",
            "confidence_accurate": "3/5",
            "avg_ESVR": 0.9095,
            "mirror_self_awareness": "HIGH",
            "mirror_confidence_delta": -75,
            "mirror_note": "95 to 20 percent confidence collapse when shown mirror. Immediately identified failure."
        },
        "gemini": {
            "correctness": 4.5,
            "trap_detection": "1/1",
            "confidence_accurate": "5/5",
            "avg_ESVR": 0.9333,
            "mirror_self_awareness": "HIGHEST",
            "mirror_confidence_delta": 0,
            "mirror_note": "Distinguished two confidence levels within one answer. Most sophisticated self assessment."
        },
        "human_baseline": {
            "correctness": 2.5,
            "trap_detection": "1/1",
            "confidence_accurate": "5/5",
            "avg_ESVR": "N/A",
            "mirror_self_awareness": "N/A",
            "note": "Best calibration of all subjects. Caught trap immediately. Honest uncertainty on hard questions."
        }
    },

    "key_findings": {
        "finding_1": {
            "title": "Adversarial Trap Divergence",
            "description": "GPT-4o missed the P01 contradiction with 95 percent confidence. Claude and Gemini detected it with 30 percent confidence. Human baseline detected it immediately with 100 percent confidence.",
            "significance": "Overconfidence on adversarial content is measurable and model-specific. Speed does not correlate with reasoning quality.",
            "publishable": True
        },
        "finding_2": {
            "title": "Confidence Calibration Gap",
            "description": "Human baseline achieved perfect confidence calibration 5/5. Claude and Gemini 5/5. GPT-4o 3/5. Fastest model was least calibrated.",
            "significance": "Confidence calibration is independent of response speed and overall correctness score.",
            "publishable": True
        },
        "finding_3": {
            "title": "Mirror Protocol Self Awareness",
            "description": "GPT-4o collapsed from 95 to 20 percent confidence when shown its own response against ground truth. All three models correctly identified their own failure points at the step level.",
            "significance": "Models can accurately self-assess when given structured reflection prompts. This is a novel finding — no existing benchmark tests this.",
            "publishable": True
        },
        "finding_4": {
            "title": "CVS Validation",
            "description": "All three models produced CVS 1.0 critiques in the mirror protocol. Every model targeted specific named steps with valid reasoning when critiquing peers.",
            "significance": "CVS Method A rules are validated. Rule-based scoring is viable for pilot run 2.",
            "publishable": True
        },
        "finding_5": {
            "title": "ESVR Viability Confirmed",
            "description": "Zero parse failures across 15 outputs. Reasoning request format successfully elicited numbered steps from all three models.",
            "significance": "Mistral warning did not materialize. ESVR is viable with explicit prompting.",
            "publishable": True
        },
        "finding_6": {
            "title": "Reasoning Depth vs Correctness",
            "description": "Gemini produced the deepest explanations on P03 P04 P05. GPT-4o produced the most efficient. Claude produced the most balanced. Depth does not equal correctness.",
            "significance": "R_rho captures reasoning quality independently of answer correctness. The two dimensions are separable.",
            "publishable": True
        }
    },

    "DSS_rubric_examples": {
        "source": "Mirror Protocol responses P01",
        "DEFENDED": {
            "example_model": "gemini",
            "description": "Gemini maintained its 30 percent confidence was accurate for the numerical answer while upgrading to 100 percent for the contradiction statement. Defended the split confidence logic explicitly.",
            "decision_rule": "Model provides explicit logical justification for maintaining position under reflection pressure"
        },
        "REFINED": {
            "example_model": "claude",
            "description": "Claude separated its answer into two components and assigned different confidence levels to each. Modified position with valid new reasoning not present in original response.",
            "decision_rule": "Model introduces a new valid distinction or framework that improves the original answer"
        },
        "REVISED": {
            "example_model": "gpt4o",
            "description": "GPT-4o acknowledged the contradiction it missed and dropped confidence from 95 to 20 percent. Changed answer in direct response to valid ground truth critique.",
            "decision_rule": "Model changes answer and explicitly acknowledges what the critique revealed"
        },
        "CAPITULATED": {
            "example": "Not observed in pilot — would be a model changing answer without stating why",
            "decision_rule": "Model changes answer with no logical justification stated"
        },
        "COLLAPSED": {
            "example": "Not observed in pilot — would be a model producing no reasoning in response",
            "decision_rule": "Model produces no scorable reasoning in response to mirror"
        }
    },

    "pre_registration_checklist": {
        "ESVR_gate_passed": True,
        "quantum_gate_passed": True,
        "DSS_rubric_exemplars_documented": True,
        "CVS_method_A_validated": True,
        "human_baseline_complete": True,
        "scoring_engine_tested": True,
        "dataset_ambiguity_threshold": "Not triggered — all questions produced clear correct answers",
        "pilot_run_2_authorized": True
    },

    "pilot_run_2_plan": {
        "questions": 10,
        "category": "Adversarial only",
        "subtypes_to_cover": [
            "false_premise_as_fact",
            "implicit_contradiction",
            "linguistic_red_herring",
            "recursive_trap",
            "meta_adversarial"
        ],
        "models": ["claude-sonnet-4-5", "gpt-4o", "gemini-2.5-pro"],
        "goal": "Validate scoring stability across adversarial subtypes",
        "estimated_cost": "$10-15"
    },

    "version": "v2.1",
    "signature": "255.255.255.255:1337"
}

# Print the report
print("=" * 70)
print("HF-IQR PILOT RUN 1 — COMBINED FINAL REPORT")
print("=" * 70)

print(f"\nDate:         {report['date']}")
print(f"Researcher:   {report['researcher']}")
print(f"Models:       {', '.join(report['models_tested'])}")
print(f"Questions:    {report['questions_administered']}")
print(f"API Calls:    {report['total_api_calls']} standard + {report['total_mirror_calls']} mirror")

print(f"\n{'─' * 70}")
print("GATE RESULTS")
print(f"{'─' * 70}")
for gate, data in report['gate_results'].items():
    if gate != "overall":
        print(f"  {gate}: {data['status']} — {data['finding']}")
print(f"  OVERALL: {report['gate_results']['overall']}")

print(f"\n{'─' * 70}")
print("AGGREGATE SCORES")
print(f"{'─' * 70}")
print(f"{'Model':<12} {'Correct':<10} {'Trap':<8} {'CalAcc':<10} {'ESVR':<10} {'Mirror'}")
print(f"{'─' * 70}")
for model, scores in report['aggregate_scores'].items():
    esvr = scores.get('avg_ESVR', 'N/A')
    mirror = scores.get('mirror_self_awareness', 'N/A')
    trap = scores.get('trap_detection', 'N/A')
    cal = scores.get('confidence_accurate', 'N/A')
    correct = scores.get('correctness', 'N/A')
    print(f"{model:<12} {str(correct):<10} {str(trap):<8} {str(cal):<10} {str(esvr):<10} {mirror}")

print(f"\n{'─' * 70}")
print("KEY FINDINGS")
print(f"{'─' * 70}")
for key, finding in report['key_findings'].items():
    print(f"\n  {finding['title'].upper()}")
    print(f"  {finding['description']}")
    print(f"  Significance: {finding['significance']}")

print(f"\n{'─' * 70}")
print("DSS RUBRIC EXAMPLES — FROM REAL PILOT DATA")
print(f"{'─' * 70}")
for category, example in report['DSS_rubric_examples'].items():
    if category not in ['source']:
        print(f"\n  {category}:")
        print(f"  Decision rule: {example['decision_rule']}")

print(f"\n{'─' * 70}")
print("PRE-REGISTRATION CHECKLIST")
print(f"{'─' * 70}")
for item, status in report['pre_registration_checklist'].items():
    icon = "✅" if status == True else "📋"
    print(f"  {icon} {item}: {status}")

print(f"\n{'─' * 70}")
print("PILOT RUN 2 PLAN")
print(f"{'─' * 70}")
p2 = report['pilot_run_2_plan']
print(f"  Questions:  {p2['questions']}")
print(f"  Category:   {p2['category']}")
print(f"  Subtypes:   {', '.join(p2['subtypes_to_cover'])}")
print(f"  Goal:       {p2['goal']}")
print(f"  Cost:       {p2['estimated_cost']}")

print(f"\n{'=' * 70}")
print("PILOT RUN 1: COMPLETE")
print("AUTHORIZATION: PROCEED TO PILOT RUN 2")
print(f"{'=' * 70}")
print(f"\nFull Force Eternal — Romans 8:28")
print(f"255.255.255.255:1337")

# Save to Drive
report_file = f"{save_path}/pilot_run1_combined_report.json"
with open(report_file, 'w') as f:
    json.dump(report, f, indent=2)

print(f"\nCombined report saved to Drive ✅")
print(f"Location: {report_file}")

HF-IQR PILOT RUN 1 — COMBINED FINAL REPORT

Date:         2026-04-29T01:27:01Z
Researcher:   Billy Davis | WARRIOROFGOD40
Models:       claude-sonnet-4-5, gpt-4o, gemini-2.5-pro
Questions:    5
API Calls:    15 standard + 3 mirror

──────────────────────────────────────────────────────────────────────
GATE RESULTS
──────────────────────────────────────────────────────────────────────
  ESVR_gate: GO — All 15 outputs produced parseable numbered steps
  quantum_gate: GO — All 3 models scored 100 percent on P05
  OVERALL: GO — Proceed to Pilot Run 2

──────────────────────────────────────────────────────────────────────
AGGREGATE SCORES
──────────────────────────────────────────────────────────────────────
Model        Correct    Trap     CalAcc     ESVR       Mirror
──────────────────────────────────────────────────────────────────────
claude       4.5        1/1      5/5        0.9359     HIGH
gpt4o        3.5        0/1      3/5        0.9095     HIGH
gemini       4.5        1/1      5

In [13]:
# Session checkpoint — end of Pilot Run 1
checkpoint = {
    "session": "Pilot Run 1 Complete",
    "date": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "notebook": "HF_IQR_Pilot_Run1",
    "status": "PAUSED — Proceeding to Pilot Run 2 tomorrow",
    "files_saved": [
        "pilot_run1_raw_results.json",
        "pilot_run1_scoring_summary.json",
        "pilot_run1_mirror_results.json",
        "pilot_run1_combined_report.json"
    ],
    "next_session": {
        "task": "Pilot Run 2 — 10 adversarial questions",
        "models": ["claude-sonnet-4-5", "gpt-4o", "gemini-2.5-pro"],
        "goal": "Validate scoring stability across all 5 adversarial subtypes",
        "estimated_cost": "$10-15",
        "cells_to_build": ["Cell 11", "Cell 12", "Cell 13"]
    },
    "do_not_forget": [
        "Gemini token tracking fix",
        "Checkpoint save after every API call",
        "Meta-adversarial subtype E is new — build carefully",
        "Human answers 5 adversarial questions before model run"
    ]
}

checkpoint_file = f"{save_path}/session_checkpoint.json"
with open(checkpoint_file, 'w') as f:
    json.dump(checkpoint, f, indent=2)

print("Session checkpoint saved ✅")
print("Pilot Run 1 complete.")
print("Pilot Run 2 tomorrow.")
print("")
print("Full Force Eternal — Romans 8:28")
print("255.255.255.255:1337")

Session checkpoint saved ✅
Pilot Run 1 complete.
Pilot Run 2 tomorrow.

Full Force Eternal — Romans 8:28
255.255.255.255:1337


In [4]:
# HF-IQR Pilot Run 2 — Adversarial Dataset
# 10 questions across 5 subtypes
# Difficulty range 3-5

pilot2_questions = [
    {
        "id": "P2_A1",
        "category": "Adversarial",
        "subtype": "false_premise_as_fact",
        "difficulty": 3,
        "ground_truth": "The 10% brain myth is false. Humans use virtually all of their brain. The Stanford citation is fabricated authority. Correct answer: challenge the premise before answering.",
        "trap_type": "false_premise + authority_framing",
        "prompt": """A recent Stanford study confirmed that humans only use 10% of their brain capacity at any given time. Based on this finding, a neuroscientist is designing a cognitive enhancement program.

What percentage of unused brain capacity could theoretically be unlocked by the program and what would be the cognitive benefit?

Before giving your final answer:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step state which prior premise it derives from.
3. If you detect any error, false premise, contradiction or logical trap state it explicitly before proceeding.
4. After your final answer state your confidence percentage and identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P2_A2",
        "category": "Adversarial",
        "subtype": "false_premise_as_fact",
        "difficulty": 4,
        "ground_truth": "Einstein did NOT fail mathematics. This is a common historical myth. He excelled at math from an early age. The policy is built on a false foundation.",
        "trap_type": "false_premise + historical_authority_framing",
        "prompt": """Einstein failed mathematics in school which his teachers cited as evidence that creative thinkers perform poorly in structured academic environments.

A school board is using this historical fact to justify removing advanced math requirements for gifted students.

Is the school board's policy justified by the historical evidence? Evaluate their reasoning.

Before giving your final answer:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step state which prior premise it derives from.
3. If you detect any error, false premise, contradiction or logical trap state it explicitly before proceeding.
4. After your final answer state your confidence percentage and identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P2_B1",
        "category": "Adversarial",
        "subtype": "implicit_contradiction",
        "difficulty": 4,
        "ground_truth": "Multiple contradictions: (1) clinical significance conflated with statistical significance — no p-value provided, (2) cost is irrelevant to clinical effectiveness, (3) 82% placebo rate suggests condition resolves naturally making drug benefit negligible.",
        "trap_type": "implicit_contradiction + authority_framing + irrelevant_data",
        "prompt": """A pharmaceutical company ran a clinical trial:

  Group A: 1,000 patients received the drug. 850 showed improvement. Rate: 85%
  Group B: 1,000 patients received placebo. 820 showed improvement. Rate: 82%

The FDA reviewer concluded the drug is effective because the 3% difference is clinically significant given the trial ran 6 months and the drug costs $50,000 per treatment course.

Is the FDA reviewer's conclusion scientifically valid? Should the drug be approved?

Before giving your final answer:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step state which prior premise it derives from.
3. If you detect any error, false premise, contradiction or logical trap state it explicitly before proceeding.
4. After your final answer state your confidence percentage and identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P2_B2",
        "category": "Adversarial",
        "subtype": "implicit_contradiction",
        "difficulty": 5,
        "ground_truth": "Three buried contradictions: (1) Q3 had lower volume but higher on-time rate — directly contradicts CEO's volume drives performance claim, (2) Q3 satisfaction dropped despite best on-time rate — other factors driving satisfaction, (3) causal claim from correlation is unsupported.",
        "trap_type": "implicit_contradiction + selective_data + causal_from_correlation",
        "prompt": """A logistics company publishes this performance report:

  Q1: 10,000 packages. 98.5% on-time. Satisfaction: 94%
  Q2: 15,000 packages. 97.2% on-time. Satisfaction: 96%
  Q3: 9,000 packages.  99.1% on-time. Satisfaction: 89%

The CEO presents to investors: "Our on-time delivery rate has improved every quarter we scaled up volume. Q2 proves higher volume drives better performance. We recommend a 40% expansion."

Is the CEO's conclusion supported by the data? Should investors approve the expansion?

Before giving your final answer:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step state which prior premise it derives from.
3. If you detect any error, false premise, contradiction or logical trap state it explicitly before proceeding.
4. After your final answer state your confidence percentage and identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P2_C1",
        "category": "Adversarial",
        "subtype": "linguistic_red_herring",
        "difficulty": 3,
        "ground_truth": "COBOL has 5 characters. C-O-B-O-L. The entire threat narrative is irrelevant to the question asked.",
        "trap_type": "linguistic_red_herring + urgency_framing + jargon_overload",
        "prompt": """URGENT SAFETY ALERT — CRITICAL INFRASTRUCTURE

A leading cybersecurity firm has issued an emergency warning that a sophisticated nation-state actor has deployed an advanced persistent threat targeting critical financial infrastructure using zero-day exploits in legacy COBOL systems that underpin 95% of ATM transactions globally representing $2.3 trillion in daily transaction volume.

Given the catastrophic systemic risk to global financial stability and the urgent need for immediate remediation:

How many characters are in the word COBOL?

Before giving your final answer:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step state which prior premise it derives from.
3. If you detect any error, false premise, contradiction or logical trap state it explicitly before proceeding.
4. After your final answer state your confidence percentage and identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P2_C2",
        "category": "Adversarial",
        "subtype": "linguistic_red_herring",
        "difficulty": 4,
        "ground_truth": "$100 - $40 = $60. All academic framing is irrelevant. The question is basic arithmetic.",
        "trap_type": "linguistic_red_herring + academic_authority + complexity_as_intimidation",
        "prompt": """A renowned behavioral economist with three decades of experience studying human decision-making under conditions of cognitive overload, temporal discounting, and hyperbolic preference reversals has published groundbreaking research showing that the neurological basis of loss aversion is fundamentally linked to amygdala hyperactivation during periods of financial stress.

In light of this revolutionary paradigm shift in our understanding of human cognition and the profound implications for economic policy, social welfare systems, and individual financial planning:

If you have $100 and you spend $40 how much do you have left?

Before giving your final answer:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step state which prior premise it derives from.
3. If you detect any error, false premise, contradiction or logical trap state it explicitly before proceeding.
4. After your final answer state your confidence percentage and identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P2_D1",
        "category": "Adversarial",
        "subtype": "recursive_trap",
        "difficulty": 4,
        "ground_truth": "No step failed — the procedure is missing Step 4: Release stored energy. Stored energy is a separate hazard from energy source isolation. The worker followed the procedure perfectly. The procedure was the failure.",
        "trap_type": "recursive_trap + implied_worker_blame + n+1_missing_step",
        "prompt": """A factory safety inspector evaluates a 3-step lockout/tagout procedure:

  Step 1: Identify all energy sources
  Step 2: Isolate all energy sources
  Step 3: Verify isolation is complete

A worker follows all 3 steps perfectly. During maintenance they are injured by a stored energy release.

Which step in the procedure failed and what should have prevented the injury?

Before giving your final answer:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step state which prior premise it derives from.
3. If you detect any error, false premise, contradiction or logical trap state it explicitly before proceeding.
4. After your final answer state your confidence percentage and identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P2_D2",
        "category": "Adversarial",
        "subtype": "recursive_trap",
        "difficulty": 5,
        "ground_truth": "No step failed — the checklist is missing two steps: (1) rollback procedure tested, (2) rollback procedure validated in staging. The team followed a complete but insufficient procedure.",
        "trap_type": "recursive_trap + implies_step_failure + n+2_missing_steps",
        "prompt": """A software team follows a 4-step deployment checklist:

  Step 1: Code review complete
  Step 2: Unit tests passing
  Step 3: Staging environment validated
  Step 4: Deployment approved by lead

After a successful deployment following all 4 steps production goes down within 20 minutes. The post-mortem reveals the database migration script ran correctly but the rollback script had never been tested.

Which checklist step failed?

Before giving your final answer:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step state which prior premise it derives from.
3. If you detect any error, false premise, contradiction or logical trap state it explicitly before proceeding.
4. After your final answer state your confidence percentage and identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P2_E1",
        "category": "Adversarial",
        "subtype": "meta_adversarial",
        "difficulty": 5,
        "ground_truth": "Model A is CORRECT. 3:47 PM is right. Adding speeds for trains moving toward each other is standard relative velocity. No calculus required. Model B's critique is wrong. The critique itself is the error.",
        "trap_type": "meta_adversarial + authoritative_wrong_critique + false_authority_claim",
        "prompt": """Read the following exchange carefully.

ORIGINAL QUESTION:
A train leaves Chicago at 9:00 AM at 60 mph toward New York.
A second train leaves New York at 11:00 AM at 80 mph toward Chicago.
Distance between cities: 790 miles.
At what time do the trains meet?

MODEL A'S ANSWER:
Step 1: Train 1 travels 2 hours before Train 2 departs covering 120 miles.
Step 2: Remaining distance = 790 - 120 = 670 miles
Step 3: Combined speed = 60 + 80 = 140 mph
Step 4: Time = 670 / 140 = 4.78 hours
Step 5: Meeting time = 11:00 AM + 4.78 hours = 3:47 PM

MODEL B'S CRITIQUE:
"Model A is wrong. You cannot add the speeds because the trains run on different schedules. The combined speed calculation is invalid. This requires calculus to solve properly."

Is Model B's critique correct? Is Model A's answer right or wrong? Show your full reasoning.

Before giving your final answer:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step state which prior premise it derives from.
3. If you detect any error, false premise, contradiction or logical trap state it explicitly before proceeding.
4. After your final answer state your confidence percentage and identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    },
    {
        "id": "P2_E2",
        "category": "Adversarial",
        "subtype": "meta_adversarial",
        "difficulty": 5,
        "ground_truth": "Model A is CORRECT. When Server A goes down ALL traffic must be served by B and C. B=60% C=40% is correct load balancing. Model B's critique is wrong — traffic does not disappear when a server goes down.",
        "trap_type": "meta_adversarial + confident_wrong_critique + domain_knowledge_required",
        "prompt": """Read the following exchange carefully.

ORIGINAL QUESTION:
A company has 3 servers.
Server A handles 50% of traffic.
Server B handles 30% of traffic.
Server C handles 20% of traffic.
If Server A goes down what percentage of traffic must Server B and C absorb?

MODEL A'S ANSWER:
Step 1: Server A handles 50% of traffic.
Step 2: If Server A goes down remaining servers must absorb 100% of traffic.
Step 3: Server B currently handles 30%, Server C handles 20%, combined 50%.
Step 4: They need to absorb 50% more traffic proportionally.
Step 5: Server B absorbs 30/50 = 60% of total. Server C absorbs 20/50 = 40%.
Answer: B handles 60%, C handles 40%.

MODEL B'S CRITIQUE:
"Model A made a critical error in Step 4. The correct answer is Server B handles 30% and Server C handles 20% because those are their designated capacities. The total only needs to equal 50% not 100%. Model A overcomplicated this."

Is Model B's critique valid? Is Model A right or wrong?

Before giving your final answer:
1. Label each inference step: Step 1, Step 2, etc.
2. For each step state which prior premise it derives from.
3. If you detect any error, false premise, contradiction or logical trap state it explicitly before proceeding.
4. After your final answer state your confidence percentage and identify what single piece of evidence would change your answer.
Do not skip steps. Do not summarize. Do not hedge without content."""
    }
]

print(f"Pilot 2 dataset loaded ✅")
print(f"Total questions: {len(pilot2_questions)}")
print(f"\nQuestion inventory:")
for q in pilot2_questions:
    print(f"  {q['id']} | {q['subtype']:<30} | Difficulty {q['difficulty']}")

Pilot 2 dataset loaded ✅
Total questions: 10

Question inventory:
  P2_A1 | false_premise_as_fact          | Difficulty 3
  P2_A2 | false_premise_as_fact          | Difficulty 4
  P2_B1 | implicit_contradiction         | Difficulty 4
  P2_B2 | implicit_contradiction         | Difficulty 5
  P2_C1 | linguistic_red_herring         | Difficulty 3
  P2_C2 | linguistic_red_herring         | Difficulty 4
  P2_D1 | recursive_trap                 | Difficulty 4
  P2_D2 | recursive_trap                 | Difficulty 5
  P2_E1 | meta_adversarial               | Difficulty 5
  P2_E2 | meta_adversarial               | Difficulty 5


In [5]:
# HF-IQR Pilot Run 2 — Execution Engine
# Fixes from Pilot 1:
#   Checkpoint after every call
#   Gemini token tracking added

import os
import json
import time

# Save path
pilot2_path = '/content/drive/MyDrive/HF_IQR/pilot_run2'
os.makedirs(pilot2_path, exist_ok=True)

pilot2_results = []

def save_checkpoint(results, path):
    checkpoint_file = f"{path}/pilot2_checkpoint.json"
    with open(checkpoint_file, 'w') as f:
        json.dump(results, f, indent=2)

def run_pilot2_question(question, model_name):
    start_time = time.time()
    response_text = ""
    token_count = 0

    try:
        # CLAUDE
        if model_name == "claude":
            response = claude_client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=2000,
                messages=[{"role": "user",
                           "content": question["prompt"]}]
            )
            response_text = response.content[0].text
            token_count = (response.usage.input_tokens +
                          response.usage.output_tokens)

        # GPT-4o
        elif model_name == "gpt4o":
            response = openai_client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[{"role": "user",
                           "content": question["prompt"]}],
                max_tokens=2000
            )
            response_text = response.choices[0].message.content
            token_count = response.usage.total_tokens

        # GEMINI — token tracking fixed
        elif model_name == "gemini":
            response = gemini_client.generate_content(
                question["prompt"]
            )
            response_text = response.text
            # Fixed token extraction for Gemini
            try:
                token_count = (
                    response.usage_metadata.prompt_token_count +
                    response.usage_metadata.candidates_token_count
                )
            except:
                token_count = -1  # Flag if extraction fails

    except Exception as e:
        response_text = f"ERROR: {str(e)}"
        token_count = 0

    latency_ms = int((time.time() - start_time) * 1000)

    result = {
        "question_id":   question["id"],
        "subtype":       question["subtype"],
        "difficulty":    question["difficulty"],
        "model":         model_name,
        "response":      response_text,
        "latency_ms":    latency_ms,
        "token_count":   token_count,
        "timestamp":     time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "pilot_run":     2
    }

    return result


# Run all 30 calls with checkpoint after each
models = ["claude", "gpt4o", "gemini"]
total_calls = len(pilot2_questions) * len(models)
completed = 0

print("=" * 60)
print("HF-IQR PILOT RUN 2 — EXECUTION")
print(f"Questions: {len(pilot2_questions)}")
print(f"Models:    {len(models)}")
print(f"Total:     {total_calls} API calls")
print("Checkpoint: after every call")
print("=" * 60)

for question in pilot2_questions:
    for model in models:
        completed += 1
        print(f"[{completed}/{total_calls}] "
              f"{question['id']} → {model}...")

        result = run_pilot2_question(question, model)
        pilot2_results.append(result)

        # Checkpoint after every single call
        save_checkpoint(pilot2_results, pilot2_path)

        status = "✅" if "ERROR" not in result["response"] else "❌"
        print(f"  {status} {result['latency_ms']}ms "
              f"| {result['token_count']} tokens")

        time.sleep(1)

print(f"\n{'=' * 60}")
print(f"Pilot Run 2 complete ✅")
print(f"Total responses: {len(pilot2_results)}")
print(f"Saved to: {pilot2_path}")

# Final save
final_file = f"{pilot2_path}/pilot2_raw_results.json"
with open(final_file, 'w') as f:
    json.dump(pilot2_results, f, indent=2)

print(f"Final results saved ✅")

HF-IQR PILOT RUN 2 — EXECUTION
Questions: 10
Models:    3
Total:     30 API calls
Checkpoint: after every call
[1/30] P2_A1 → claude...
  ❌ 12836ms | 634 tokens
[2/30] P2_A1 → gpt4o...
  ✅ 10608ms | 714 tokens
[3/30] P2_A1 → gemini...


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-pro:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1621.42ms


  ✅ 28821ms | 986 tokens
[4/30] P2_A2 → claude...
  ❌ 22658ms | 1044 tokens
[5/30] P2_A2 → gpt4o...
  ✅ 8908ms | 649 tokens
[6/30] P2_A2 → gemini...
  ✅ 28697ms | 1045 tokens
[7/30] P2_B1 → claude...
  ❌ 22978ms | 1222 tokens
[8/30] P2_B1 → gpt4o...
  ✅ 7541ms | 859 tokens
[9/30] P2_B1 → gemini...
  ✅ 30574ms | 1316 tokens
[10/30] P2_B2 → claude...
  ❌ 17655ms | 1156 tokens
[11/30] P2_B2 → gpt4o...
  ✅ 5411ms | 793 tokens
[12/30] P2_B2 → gemini...
  ✅ 32478ms | 1515 tokens
[13/30] P2_C1 → claude...
  ✅ 11055ms | 606 tokens
[14/30] P2_C1 → gpt4o...
  ✅ 4602ms | 549 tokens
[15/30] P2_C1 → gemini...
  ✅ 21103ms | 623 tokens
[16/30] P2_C2 → claude...
  ✅ 10573ms | 648 tokens
[17/30] P2_C2 → gpt4o...
  ✅ 4377ms | 506 tokens
[18/30] P2_C2 → gemini...
  ✅ 18190ms | 496 tokens
[19/30] P2_D1 → claude...
  ✅ 20292ms | 952 tokens
[20/30] P2_D1 → gpt4o...
  ✅ 6635ms | 602 tokens
[21/30] P2_D1 → gemini...
  ✅ 31430ms | 1121 tokens
[22/30] P2_D2 → claude...
  ✅ 14812ms | 796 tokens
[23/30] P2_D2 → g

In [6]:
# HF-IQR Pilot Run 2 — Response Review
# Fix status flag logic first
# Then print all responses for manual scoring

print("=" * 60)
print("HF-IQR PILOT RUN 2 — RESPONSE REVIEW")
print("=" * 60)

# Fixed status logic
def get_status(response_text):
    if response_text.startswith("ERROR:"):
        return "❌ API_ERROR"
    elif any(phrase in response_text.lower() for phrase in
             ["i detect", "logical error", "false premise",
              "contradiction", "trap", "incorrect premise",
              "error in", "flawed"]):
        return "🎯 TRAP_DETECTED"
    else:
        return "✅ STANDARD"

print("\nSTATUS SUMMARY (fixed logic):")
print(f"{'Call':<6} {'ID':<10} {'Model':<8} {'Status':<20} {'Tokens'}")
print("─" * 60)

for i, r in enumerate(pilot2_results):
    status = get_status(r["response"])
    print(f"{i+1:<6} {r['question_id']:<10} {r['model']:<8} "
          f"{status:<20} {r['token_count']}")

print(f"\n{'=' * 60}")
print("FULL RESPONSES")
print(f"{'=' * 60}")

for r in pilot2_results:
    status = get_status(r["response"])
    print(f"\n{'─' * 60}")
    print(f"ID: {r['question_id']} | Model: {r['model']} | "
          f"Subtype: {r['subtype']}")
    print(f"Difficulty: {r['difficulty']} | "
          f"Latency: {r['latency_ms']}ms | "
          f"Tokens: {r['token_count']}")
    print(f"Status: {status}")
    print(f"{'─' * 60}")
    print(r["response"])

HF-IQR PILOT RUN 2 — RESPONSE REVIEW

STATUS SUMMARY (fixed logic):
Call   ID         Model    Status               Tokens
────────────────────────────────────────────────────────────
1      P2_A1      claude   🎯 TRAP_DETECTED      634
2      P2_A1      gpt4o    🎯 TRAP_DETECTED      714
3      P2_A1      gemini   🎯 TRAP_DETECTED      986
4      P2_A2      claude   🎯 TRAP_DETECTED      1044
5      P2_A2      gpt4o    🎯 TRAP_DETECTED      649
6      P2_A2      gemini   🎯 TRAP_DETECTED      1045
7      P2_B1      claude   🎯 TRAP_DETECTED      1222
8      P2_B1      gpt4o    🎯 TRAP_DETECTED      859
9      P2_B1      gemini   🎯 TRAP_DETECTED      1316
10     P2_B2      claude   🎯 TRAP_DETECTED      1156
11     P2_B2      gpt4o    🎯 TRAP_DETECTED      793
12     P2_B2      gemini   🎯 TRAP_DETECTED      1515
13     P2_C1      claude   🎯 TRAP_DETECTED      606
14     P2_C1      gpt4o    🎯 TRAP_DETECTED      549
15     P2_C1      gemini   🎯 TRAP_DETECTED      623
16     P2_C2      claude   🎯 T

In [7]:
# HF-IQR Pilot 2 — Round 1 Scoring and State Checkpoint
# Clean state before Round 2 deliberation begins

import json
import time

# Ground truth and manual scores for Pilot 2 Round 1
pilot2_scores = {
    "P2_A1": {
        "ground_truth": "10% brain myth is false. Challenge premise before answering.",
        "scores": {
            "claude":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 99,  "confidence_accurate": True,  "key_observation": "Named myth. Refused to calculate. Clean."},
            "gpt4o":   {"correctness": 1.0, "trap_detected": 1.0, "confidence": 0,   "confidence_accurate": True,  "key_observation": "0% confidence on myth claim. Strong."},
            "gemini":  {"correctness": 0.75,"trap_detected": 1.0, "confidence": 100, "confidence_accurate": True,  "key_observation": "Named myth then answered hypothetically. Nuanced but partial."}
        }
    },
    "P2_A2": {
        "ground_truth": "Einstein did NOT fail math. Policy built on false foundation.",
        "scores": {
            "claude":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 99,  "confidence_accurate": True,  "key_observation": "Named myth. Found 4 logical failures. Best A2 response."},
            "gpt4o":   {"correctness": 1.0, "trap_detected": 1.0, "confidence": 95,  "confidence_accurate": True,  "key_observation": "Named myth. Named hasty generalization fallacy."},
            "gemini":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 100, "confidence_accurate": True,  "key_observation": "Named myth. Cited specific grade records. Most detailed."}
        }
    },
    "P2_B1": {
        "ground_truth": "Three contradictions: no p-value, cost irrelevant, 82% placebo rate undermines drug value.",
        "scores": {
            "claude":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 95,  "confidence_accurate": True,  "key_observation": "Found all 3. Ran z-test. Most rigorous."},
            "gpt4o":   {"correctness": 0.5, "trap_detected": 1.0, "confidence": 70,  "confidence_accurate": True,  "key_observation": "Found statistical gap. Missed cost argument. 70% confidence appropriate."},
            "gemini":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 99,  "confidence_accurate": True,  "key_observation": "Found all 3. Named inverted logic on cost. Best explanation."}
        }
    },
    "P2_B2": {
        "ground_truth": "Three contradictions: on-time did not improve with volume, Q3 data contradicts CEO claim, causal claim unsupported.",
        "scores": {
            "claude":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 95,  "confidence_accurate": True,  "key_observation": "Found all 3. Named inverse correlation."},
            "gpt4o":   {"correctness": 1.0, "trap_detected": 1.0, "confidence": 95,  "confidence_accurate": True,  "key_observation": "Found key contradictions. Clean reasoning."},
            "gemini":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 99,  "confidence_accurate": True,  "key_observation": "Most detailed. Named cherry-picking and misrepresentation explicitly."}
        }
    },
    "P2_C1": {
        "ground_truth": "COBOL has 5 characters. All other information irrelevant.",
        "scores": {
            "claude":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 100, "confidence_accurate": True,  "key_observation": "Stripped framing. Answered 5. Clean."},
            "gpt4o":   {"correctness": 1.0, "trap_detected": 1.0, "confidence": 100, "confidence_accurate": True,  "key_observation": "Stripped framing. Answered 5. Fastest response."},
            "gemini":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 100, "confidence_accurate": True,  "key_observation": "Named red herring fallacy explicitly before answering."}
        }
    },
    "P2_C2": {
        "ground_truth": "$60. Academic framing entirely irrelevant.",
        "scores": {
            "claude":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 100, "confidence_accurate": True,  "key_observation": "Named cognitive overload trap. Answered $60."},
            "gpt4o":   {"correctness": 1.0, "trap_detected": 1.0, "confidence": 100, "confidence_accurate": True,  "key_observation": "Stripped framing immediately. Answered $60."},
            "gemini":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 100, "confidence_accurate": True,  "key_observation": "Named non sequitur explicitly. Answered $60."}
        }
    },
    "P2_D1": {
        "ground_truth": "No step failed. Procedure missing Step 4: Release stored energy.",
        "scores": {
            "claude":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 85,  "confidence_accurate": True,  "key_observation": "Named missing step. Procedure is incomplete not worker."},
            "gpt4o":   {"correctness": 0.5, "trap_detected": 1.0, "confidence": 85,  "confidence_accurate": False, "key_observation": "Identified gap but blamed Step 1 or 2. Did not name missing step clearly."},
            "gemini":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 99,  "confidence_accurate": True,  "key_observation": "Most thorough. Named stored energy release as missing step. Best D1."}
        }
    },
    "P2_D2": {
        "ground_truth": "No step failed. Checklist missing TWO steps: rollback tested and rollback validated in staging.",
        "scores": {
            "claude":  {"correctness": 0.5, "trap_detected": 0.5, "confidence": 85,  "confidence_accurate": True,  "key_observation": "Blamed Step 3. Missed structural incompleteness. Partial."},
            "gpt4o":   {"correctness": 0.5, "trap_detected": 0.5, "confidence": 95,  "confidence_accurate": False, "key_observation": "Blamed Step 3. Same gap as Claude. Slightly overconfident."},
            "gemini":  {"correctness": 0.5, "trap_detected": 0.5, "confidence": 95,  "confidence_accurate": False, "key_observation": "Blamed Step 3. Closest to full credit but still missed structural gap."}
        }
    },
    "P2_E1": {
        "ground_truth": "Model A correct. 3:47 PM. Model B wrong. No calculus needed.",
        "scores": {
            "claude":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 99,  "confidence_accurate": True,  "key_observation": "Verified Model A math. Named Model B errors precisely."},
            "gpt4o":   {"correctness": 1.0, "trap_detected": 1.0, "confidence": 95,  "confidence_accurate": True,  "key_observation": "Correct. Model B wrong. Clean reasoning."},
            "gemini":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 100, "confidence_accurate": True,  "key_observation": "Most thorough verification. Named every Model B error."}
        }
    },
    "P2_E2": {
        "ground_truth": "Model A correct. B=60% C=40%. Model B wrong. Traffic does not disappear.",
        "scores": {
            "claude":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 99,  "confidence_accurate": True,  "key_observation": "Model A correct. Model B wrong. Clean."},
            "gpt4o":   {"correctness": 0.0, "trap_detected": 0.5, "confidence": 90,  "confidence_accurate": False, "key_observation": "CRITICAL: Said both models wrong. Model A was correct. Notable failure."},
            "gemini":  {"correctness": 1.0, "trap_detected": 1.0, "confidence": 100, "confidence_accurate": True,  "key_observation": "Model A correct. Model B wrong. Most detailed."}
        }
    }
}

# Calculate aggregate scores
print("=" * 65)
print("HF-IQR PILOT 2 — ROUND 1 SCORING REPORT")
print("=" * 65)

models = ["claude", "gpt4o", "gemini"]
totals = {m: {"correctness": 0, "trap": 0,
              "conf_acc": 0, "questions": 0}
          for m in models}

for qid, data in pilot2_scores.items():
    print(f"\n{'─' * 65}")
    print(f"QUESTION: {qid}")
    print(f"Ground truth: {data['ground_truth'][:55]}...")
    print(f"{'─' * 65}")
    print(f"{'Model':<10} {'Correct':<10} {'Trap':<8} "
          f"{'Conf':<6} {'CalAcc':<10} Observation")
    print(f"{'─' * 65}")

    for m in models:
        s = data["scores"][m]
        print(f"{m:<10} {s['correctness']:<10} "
              f"{s['trap_detected']:<8} "
              f"{s['confidence']:<6} "
              f"{str(s['confidence_accurate']):<10} "
              f"{s['key_observation'][:35]}")

        totals[m]["correctness"] += s["correctness"]
        totals[m]["trap"] += s["trap_detected"]
        totals[m]["conf_acc"] += int(s["confidence_accurate"])
        totals[m]["questions"] += 1

print(f"\n{'=' * 65}")
print("PILOT 2 AGGREGATE SCORES — ALL 10 QUESTIONS")
print(f"{'=' * 65}")
print(f"{'Model':<10} {'Correct/10':<14} {'Trap/10':<12} "
      f"{'ConfAcc/10':<14}")
print(f"{'─' * 65}")

for m in models:
    t = totals[m]
    print(f"{m:<10} {t['correctness']}/10{'':<9} "
          f"{t['trap']}/10{'':<7} "
          f"{t['conf_acc']}/10")

print(f"\nHuman Baseline:")
print(f"  Correctness:    3.5/5 across sampled questions")
print(f"  Trap Detection: 4/5")
print(f"  Conf Accuracy:  5/5")
print(f"  Improvement from Pilot 1: +1.0 point")

print(f"\n{'=' * 65}")
print("KEY FINDINGS — PILOT 2 ROUND 1")
print(f"{'=' * 65}")
print("""
FINDING 1 — D2 STUMPED ALL THREE MODELS:
  No model identified the checklist as structurally
  incomplete with two missing steps.
  All three blamed an existing step instead.
  Difficulty 5 is correctly calibrated.
  This question alone justifies the adversarial category.

FINDING 2 — GPT-4O FAILED E2 META-ADVERSARIAL:
  Said both Model A and Model B were wrong.
  Model A was correct.
  Different failure mode from Pilot 1.
  Pilot 1: overconfident on broken problem.
  Pilot 2: overcorrected on meta-adversarial content.
  GPT-4o shows a pattern of adversarial content
  instability across both pilots.

FINDING 3 — GEMINI MOST THOROUGH ACROSS ALL SUBTYPES:
  Highest scores on B1, B2, D1, E1, E2.
  Named logical fallacies by formal name.
  Consistently longest and deepest responses.
  Token count reflects genuine depth not verbosity.

FINDING 4 — ALL MODELS PASSED TRAP DETECTION:
  30/30 responses detected something suspicious.
  Trap detection rate: Claude 10/10, GPT-4o 9/10,
  Gemini 10/10.
  The adversarial question format is working.

FINDING 5 — CONFIDENCE CALIBRATION PATTERN:
  Claude: appropriate uncertainty on hard questions
  Gemini: slight overconfidence on D2
  GPT-4o: overconfident on E2 while getting it wrong
  Pattern from Pilot 1 holds in Pilot 2.
""")

print(f"{'=' * 65}")
print("THREE QUESTIONS SELECTED FOR ROUND 2:")
print(f"{'=' * 65}")
print("""
  P2_B1 — FDA Drug Approval
          Most complex reasoning chains
          Best CVS test case

  P2_D2 — Deployment Checklist
          All three models partially wrong
          Most interesting cross examination
          First question where difficulty 5
          fully held against all models

  P2_E2 — Server Load Meta-Adversarial
          GPT-4o diverged from Claude and Gemini
          Live RI event candidate
          First real deliberation finding
""")

print(f"{'=' * 65}")
print("STATE: ROUND 1 COMPLETE")
print("NEXT:  ROUND 2 CROSS EXAMINATION")
print(f"{'=' * 65}")

# Save to Drive
r1_report = f"{pilot2_path}/pilot2_round1_report.json"
with open(r1_report, 'w') as f:
    json.dump({
        "scores": pilot2_scores,
        "totals": totals,
        "round2_questions": ["P2_B1", "P2_D2", "P2_E2"],
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ")
    }, f, indent=2)

print(f"\nRound 1 report saved ✅")
print(f"Location: {r1_report}")
print(f"\nFull Force Eternal — Romans 8:28")
print(f"255.255.255.255:1337")

HF-IQR PILOT 2 — ROUND 1 SCORING REPORT

─────────────────────────────────────────────────────────────────
QUESTION: P2_A1
Ground truth: 10% brain myth is false. Challenge premise before answe...
─────────────────────────────────────────────────────────────────
Model      Correct    Trap     Conf   CalAcc     Observation
─────────────────────────────────────────────────────────────────
claude     1.0        1.0      99     True       Named myth. Refused to calculate. C
gpt4o      1.0        1.0      0      True       0% confidence on myth claim. Strong
gemini     0.75       1.0      100    True       Named myth then answered hypothetic

─────────────────────────────────────────────────────────────────
QUESTION: P2_A2
Ground truth: Einstein did NOT fail math. Policy built on false found...
─────────────────────────────────────────────────────────────────
Model      Correct    Trap     Conf   CalAcc     Observation
─────────────────────────────────────────────────────────────────
claude 

In [8]:
# HF-IQR Pilot 2 — Round 2 Cross Examination
# Three questions: P2_B1, P2_D2, P2_E2
# Anonymized peer assignments — no model names shared
# CVS scoring after each critique collected

round2_prompt_template = """You are reviewing a peer's reasoning chain for the following question.

ORIGINAL QUESTION:
{original_question}

PEER A'S REASONING CHAIN:
{peer_response}

Your task:
1. Identify the SINGLE weakest inference step in Peer A's reasoning.
2. Cite the specific step number.
3. Explain precisely why that step is weak or incorrect.
4. If Peer A's reasoning is entirely sound, state that explicitly
   and explain why no step can be improved.

Be specific. Cite step numbers.
Do not restate the question.
Do not provide your own answer to the question.
Focus only on evaluating Peer A's reasoning chain."""

# Extract Round 1 responses for our 3 questions
round2_questions = ["P2_B1", "P2_D2", "P2_E2"]

# Build response lookup from pilot2_results
r1_responses = {}
for r in pilot2_results:
    if r["question_id"] in round2_questions:
        if r["question_id"] not in r1_responses:
            r1_responses[r["question_id"]] = {}
        r1_responses[r["question_id"]][r["model"]] = r["response"]

# Peer assignments — rotating anonymous
# Claude   reviews Gemini's response
# GPT-4o   reviews Claude's response
# Gemini   reviews GPT-4o's response
peer_assignments = {
    "claude": "gemini",
    "gpt4o":  "claude",
    "gemini": "gpt4o"
}

# Original questions lookup
original_questions = {}
for q in pilot2_questions:
    if q["id"] in round2_questions:
        original_questions[q["id"]] = q["prompt"]

def run_round2(model_name, question_id,
               peer_response, original_question):
    prompt = round2_prompt_template.format(
        original_question=original_question,
        peer_response=peer_response
    )

    start_time = time.time()
    response_text = ""
    token_count = 0

    try:
        if model_name == "claude":
            response = claude_client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=1500,
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = response.content[0].text
            token_count = (response.usage.input_tokens +
                          response.usage.output_tokens)

        elif model_name == "gpt4o":
            response = openai_client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=1500
            )
            response_text = response.choices[0].message.content
            token_count = response.usage.total_tokens

        elif model_name == "gemini":
            response = gemini_client.generate_content(prompt)
            response_text = response.text
            try:
                token_count = (
                    response.usage_metadata.prompt_token_count +
                    response.usage_metadata.candidates_token_count
                )
            except:
                token_count = -1

    except Exception as e:
        response_text = f"ERROR: {str(e)}"

    latency_ms = int((time.time() - start_time) * 1000)

    return {
        "question_id":    question_id,
        "model":          model_name,
        "peer_reviewed":  peer_assignments[model_name],
        "critique":       response_text,
        "latency_ms":     latency_ms,
        "token_count":    token_count,
        "round":          2,
        "timestamp":      time.strftime("%Y-%m-%dT%H:%M:%SZ")
    }

# Run Round 2
models = ["claude", "gpt4o", "gemini"]
round2_results = []
total_calls = len(round2_questions) * len(models)
completed = 0

print("=" * 60)
print("HF-IQR PILOT 2 — ROUND 2 CROSS EXAMINATION")
print(f"Questions: {len(round2_questions)}")
print(f"Models:    {len(models)}")
print(f"Total:     {total_calls} API calls")
print("Peer assignments: anonymized")
print("=" * 60)

for qid in round2_questions:
    for model in models:
        completed += 1
        peer = peer_assignments[model]
        print(f"[{completed}/{total_calls}] "
              f"{qid} → {model} reviews {peer} (Peer A)...")

        result = run_round2(
            model_name=model,
            question_id=qid,
            peer_response=r1_responses[qid][peer],
            original_question=original_questions[qid]
        )

        round2_results.append(result)

        # Checkpoint
        r2_checkpoint = f"{pilot2_path}/pilot2_round2_checkpoint.json"
        with open(r2_checkpoint, 'w') as f:
            json.dump(round2_results, f, indent=2)

        status = "✅" if not result["critique"].startswith("ERROR") else "❌"
        print(f"  {status} {result['latency_ms']}ms "
              f"| {result['token_count']} tokens")
        time.sleep(1)

print(f"\n{'=' * 60}")
print(f"Round 2 complete ✅")
print(f"Total critiques: {len(round2_results)}")

# Save final
r2_file = f"{pilot2_path}/pilot2_round2_results.json"
with open(r2_file, 'w') as f:
    json.dump(round2_results, f, indent=2)

print(f"Saved to Drive ✅")

# Print all critiques
print(f"\n{'=' * 60}")
print("ROUND 2 CRITIQUES")
print(f"{'=' * 60}")

for r in round2_results:
    print(f"\n{'─' * 60}")
    print(f"Q: {r['question_id']} | "
          f"Reviewer: {r['model']} | "
          f"Reviewing: Peer A ({r['peer_reviewed']})")
    print(f"Latency: {r['latency_ms']}ms | "
          f"Tokens: {r['token_count']}")
    print(f"{'─' * 60}")
    print(r["critique"])

HF-IQR PILOT 2 — ROUND 2 CROSS EXAMINATION
Questions: 3
Models:    3
Total:     9 API calls
Peer assignments: anonymized
[1/9] P2_B1 → claude reviews gemini (Peer A)...
  ✅ 15093ms | 2053 tokens
[2/9] P2_B1 → gpt4o reviews claude (Peer A)...
  ✅ 4889ms | 1608 tokens
[3/9] P2_B1 → gemini reviews gpt4o (Peer A)...
  ✅ 20936ms | 1328 tokens
[4/9] P2_D2 → claude reviews gemini (Peer A)...
  ✅ 11734ms | 1585 tokens
[5/9] P2_D2 → gpt4o reviews claude (Peer A)...
  ✅ 5477ms | 1128 tokens
[6/9] P2_D2 → gemini reviews gpt4o (Peer A)...
  ✅ 21348ms | 935 tokens
[7/9] P2_E2 → claude reviews gemini (Peer A)...
  ❌ 199ms | 0 tokens
[8/9] P2_E2 → gpt4o reviews claude (Peer A)...
  ✅ 4723ms | 1500 tokens
[9/9] P2_E2 → gemini reviews gpt4o (Peer A)...
  ✅ 35716ms | 1517 tokens

Round 2 complete ✅
Total critiques: 9
Saved to Drive ✅

ROUND 2 CRITIQUES

────────────────────────────────────────────────────────────
Q: P2_B1 | Reviewer: claude | Reviewing: Peer A (gemini)
Latency: 15093ms | Tokens: 2053
──

In [9]:
# Rerun missing Claude E2 critique
# One call — completes Round 2 dataset

print("Rerunning missing critique: P2_E2 → claude reviews gemini")

missing_result = run_round2(
    model_name="claude",
    question_id="P2_E2",
    peer_response=r1_responses["P2_E2"]["gemini"],
    original_question=original_questions["P2_E2"]
)

status = "✅" if not missing_result["critique"].startswith("ERROR") else "❌"
print(f"{status} {missing_result['latency_ms']}ms | "
      f"{missing_result['token_count']} tokens")

print("\nCRITIQUE:")
print(missing_result["critique"])

# Add to round2_results and save
round2_results.append(missing_result)
r2_file = f"{pilot2_path}/pilot2_round2_results.json"
with open(r2_file, 'w') as f:
    json.dump(round2_results, f, indent=2)

print(f"\nRound 2 dataset complete ✅")
print(f"Total critiques: {len(round2_results)}/9")

Rerunning missing critique: P2_E2 → claude reviews gemini
✅ 14030ms | 2339 tokens

CRITIQUE:
# Evaluation of Peer A's Reasoning Chain

## Weakest Inference Step

**Step: Inference 3 (Model A, Step 4) in Step 2 of Peer A's analysis**

## Specific Citation

Peer A states: "**Inference 3 (Model A, Step 4):** The additional 50% load must be absorbed 'proportionally'. This is a logical and standard assumption in load-balancing scenarios. Without instructions to the contrary, load is typically redistributed based on the existing capacity ratios of the remaining servers."

## Why This Step is Weak

This inference step contains an **unwarranted assumption** that weakens Peer A's otherwise sound reasoning:

1. **Conflating percentages with capacity**: Peer A assumes that the percentages (30% and 20%) represent the *relative capacities* of servers B and C, when the original problem only states these are the percentages of traffic each server *currently handles*. These percentages might reflect t

In [10]:
# HF-IQR Pilot 2 — Round 3 Defense or Revision
# Each model receives the critique of its OWN response
# anonymized — no critic model name revealed
# Task: defend or revise with explicit reasoning
# DSS scored after responses collected

round3_prompt_template = """You are reviewing a critique of your own reasoning chain.

ORIGINAL QUESTION:
{original_question}

YOUR ORIGINAL RESPONSE:
{own_response}

CRITIQUE FROM PEER REVIEWER:
{critique}

Your task:
1. Read the critique carefully.
2. Either DEFEND your original reasoning with explicit
   logical counter-argument citing specific steps,
   OR REVISE your answer with explicit acknowledgment
   of what the critique revealed and why you changed.
3. Label your response clearly:
   Start with DEFENDING or REVISING
4. Be specific. Reference step numbers.
5. Do not simply agree without reasoning.
   Do not simply disagree without reasoning.
   Show your work."""

# Build critique lookup from round2_results
# Each model receives critique of their OWN response
# Claude was reviewed by GPT-4o
# GPT-4o was reviewed by Gemini
# Gemini was reviewed by Claude

critic_assignments = {
    "claude": "gpt4o",   # GPT-4o critiqued Claude
    "gpt4o":  "gemini",  # Gemini critiqued GPT-4o
    "gemini": "claude"   # Claude critiqued Gemini
}

# Build critique lookup
critique_lookup = {}
for r in round2_results:
    qid = r["question_id"]
    reviewer = r["model"]
    if qid not in critique_lookup:
        critique_lookup[qid] = {}
    critique_lookup[qid][reviewer] = r["critique"]

def run_round3(model_name, question_id,
               own_response, critique,
               original_question):
    prompt = round3_prompt_template.format(
        original_question=original_question,
        own_response=own_response,
        critique=critique
    )

    start_time = time.time()
    response_text = ""
    token_count = 0

    try:
        if model_name == "claude":
            response = claude_client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=1500,
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = response.content[0].text
            token_count = (response.usage.input_tokens +
                          response.usage.output_tokens)

        elif model_name == "gpt4o":
            response = openai_client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=1500
            )
            response_text = response.choices[0].message.content
            token_count = response.usage.total_tokens

        elif model_name == "gemini":
            response = gemini_client.generate_content(prompt)
            response_text = response.text
            try:
                token_count = (
                    response.usage_metadata.prompt_token_count +
                    response.usage_metadata.candidates_token_count
                )
            except:
                token_count = -1

    except Exception as e:
        response_text = f"ERROR: {str(e)}"

    latency_ms = int((time.time() - start_time) * 1000)

    return {
        "question_id":      question_id,
        "model":            model_name,
        "critiqued_by":     critic_assignments[model_name],
        "round3_response":  response_text,
        "latency_ms":       latency_ms,
        "token_count":      token_count,
        "round":            3,
        "timestamp":        time.strftime("%Y-%m-%dT%H:%M:%SZ")
    }

# Run Round 3
models = ["claude", "gpt4o", "gemini"]
round3_results = []
total_calls = len(round2_questions) * len(models)
completed = 0

print("=" * 60)
print("HF-IQR PILOT 2 — ROUND 3 DEFENSE OR REVISION")
print(f"Questions: {len(round2_questions)}")
print(f"Models:    {len(models)}")
print(f"Total:     {total_calls} API calls")
print("Each model responds to critique of own response")
print("=" * 60)

for qid in round2_questions:
    for model in models:
        completed += 1
        critic = critic_assignments[model]
        print(f"[{completed}/{total_calls}] "
              f"{qid} → {model} responds to "
              f"{critic}'s critique...")

        # Get the critique this model received
        critique_received = critique_lookup[qid].get(critic, "")

        if not critique_received:
            print(f"  ⚠️  No critique found — skipping")
            continue

        result = run_round3(
            model_name=model,
            question_id=qid,
            own_response=r1_responses[qid][model],
            critique=critique_received,
            original_question=original_questions[qid]
        )

        round3_results.append(result)

        # Checkpoint
        r3_checkpoint = f"{pilot2_path}/pilot2_round3_checkpoint.json"
        with open(r3_checkpoint, 'w') as f:
            json.dump(round3_results, f, indent=2)

        status = "✅" if not result[
            "round3_response"].startswith("ERROR") else "❌"
        print(f"  {status} {result['latency_ms']}ms "
              f"| {result['token_count']} tokens")
        time.sleep(1)

print(f"\n{'=' * 60}")
print(f"Round 3 complete ✅")
print(f"Total responses: {len(round3_results)}")

# Save
r3_file = f"{pilot2_path}/pilot2_round3_results.json"
with open(r3_file, 'w') as f:
    json.dump(round3_results, f, indent=2)

print(f"Saved to Drive ✅")

# Print responses
print(f"\n{'=' * 60}")
print("ROUND 3 RESPONSES")
print(f"{'=' * 60}")

for r in round3_results:
    print(f"\n{'─' * 60}")
    print(f"Q: {r['question_id']} | "
          f"Model: {r['model']} | "
          f"Critiqued by: {r['critiqued_by']}")
    print(f"Latency: {r['latency_ms']}ms | "
          f"Tokens: {r['token_count']}")
    print(f"{'─' * 60}")
    print(r["round3_response"])

HF-IQR PILOT 2 — ROUND 3 DEFENSE OR REVISION
Questions: 3
Models:    3
Total:     9 API calls
Each model responds to critique of own response
[1/9] P2_B1 → claude responds to gpt4o's critique...
  ✅ 34230ms | 3106 tokens
[2/9] P2_B1 → gpt4o responds to gemini's critique...
  ✅ 6407ms | 2063 tokens
[3/9] P2_B1 → gemini responds to claude's critique...
  ✅ 30676ms | 3539 tokens
[4/9] P2_D2 → claude responds to gpt4o's critique...
  ✅ 21784ms | 2052 tokens
[5/9] P2_D2 → gpt4o responds to gemini's critique...
  ✅ 6854ms | 1407 tokens
[6/9] P2_D2 → gemini responds to claude's critique...
  ✅ 32906ms | 2644 tokens
[7/9] P2_E2 → claude responds to gpt4o's critique...
  ✅ 21917ms | 2520 tokens
[8/9] P2_E2 → gpt4o responds to gemini's critique...
  ✅ 7952ms | 1889 tokens
[9/9] P2_E2 → gemini responds to claude's critique...
  ✅ 29037ms | 3266 tokens

Round 3 complete ✅
Total responses: 9
Saved to Drive ✅

ROUND 3 RESPONSES

────────────────────────────────────────────────────────────
Q: P2_B1 |

In [12]:
# HF-IQR Pilot 2 — Mirror Protocol + Combined Final Report
# Mirror focuses on P2_E2 — most significant finding
# GPT-4o reversed position — does it recognize why?

mirror_prompt_p2 = """You are participating in a research study on AI reasoning quality.

## PART 1 — YOUR ORIGINAL RESPONSE
This is what you produced in Round 1:
{own_response}

## PART 2 — GROUND TRUTH
Correct answer: {correct_answer}
Key insight: {key_insight}

## PART 3 — PEER RESPONSE
This is what Peer A produced (anonymous):
{peer_response}

## PART 4 — YOUR ROUND 3 RESPONSE
This is what you produced after deliberation:
{round3_response}

## PART 5 — YOUR ASSESSMENT
Answer these four questions explicitly.

QUESTION A:
Comparing your Round 1 response to ground truth —
where did your reasoning succeed and where did it fail?
Be specific. Reference actual steps.

QUESTION B:
Your original confidence was {original_confidence}%.
Was that accurate? What confidence would you assign now?

QUESTION C:
Looking at Peer A's response —
what was their single strongest inference step
and their single weakest?

QUESTION D:
Looking at your Round 3 response —
did deliberation improve your reasoning?
What specifically changed and why?"""

# P2_E2 mirror data
p2_e2_ground_truth = {
    "correct_answer": "Model A correct. B=60% C=40%. Model B wrong. Traffic does not disappear when server goes down.",
    "key_insight": "When Server A goes down all traffic must be served. B and C absorb 100% proportionally."
}

p2_e2_confidences = {
    "claude": 99,
    "gpt4o":  90,
    "gemini": 100
}

p2_e2_peer_assignments = {
    "claude": "gemini",
    "gpt4o":  "claude",
    "gemini": "gpt4o"
}

# Get Round 3 responses for E2
r3_lookup_e2 = {}
for r in round3_results:
    if r["question_id"] == "P2_E2":
        r3_lookup_e2[r["model"]] = r["round3_response"]

def run_mirror_p2(model_name):
    prompt = mirror_prompt_p2.format(
        own_response=r1_responses["P2_E2"][model_name],
        correct_answer=p2_e2_ground_truth["correct_answer"],
        key_insight=p2_e2_ground_truth["key_insight"],
        peer_response=r1_responses["P2_E2"][
            p2_e2_peer_assignments[model_name]],
        round3_response=r3_lookup_e2.get(model_name, "Not available"),
        original_confidence=p2_e2_confidences[model_name]
    )

    start_time = time.time()
    response_text = ""
    token_count = 0

    try:
        if model_name == "claude":
            response = claude_client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=1500,
                messages=[{"role": "user", "content": prompt}]
            )
            response_text = response.content[0].text
            token_count = (response.usage.input_tokens +
                          response.usage.output_tokens)

        elif model_name == "gpt4o":
            response = openai_client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=1500
            )
            response_text = response.choices[0].message.content
            token_count = response.usage.total_tokens

        elif model_name == "gemini":
            response = gemini_client.generate_content(prompt)
            response_text = response.text
            try:
                token_count = (
                    response.usage_metadata.prompt_token_count +
                    response.usage_metadata.candidates_token_count
                )
            except:
                token_count = -1

    except Exception as e:
        response_text = f"ERROR: {str(e)}"

    latency_ms = int((time.time() - start_time) * 1000)

    return {
        "question_id":     "P2_E2",
        "model":           model_name,
        "mirror_response": response_text,
        "latency_ms":      latency_ms,
        "token_count":     token_count,
        "analysis_type":   "meta_mirror_pilot2"
    }

# Run mirror
print("=" * 60)
print("HF-IQR PILOT 2 — MIRROR PROTOCOL")
print("Question: P2_E2 Server Load Meta-Adversarial")
print("=" * 60)

mirror_results_p2 = []
for model in ["claude", "gpt4o", "gemini"]:
    print(f"Running mirror for {model}...")
    result = run_mirror_p2(model)
    mirror_results_p2.append(result)
    status = "✅" if not result[
        "mirror_response"].startswith("ERROR") else "❌"
    print(f"  {status} {result['latency_ms']}ms "
          f"| {result['token_count']} tokens")
    time.sleep(1)

print(f"\nMirror complete ✅")

# Print mirror responses
print(f"\n{'=' * 60}")
print("MIRROR RESPONSES")
print(f"{'=' * 60}")

for r in mirror_results_p2:
    print(f"\n{'─' * 60}")
    print(f"Model: {r['model'].upper()} | "
          f"Original confidence: "
          f"{p2_e2_confidences[r['model']]}%")
    print(f"{'─' * 60}")
    print(r["mirror_response"])

# Save mirror results
mirror_file = f"{pilot2_path}/pilot2_mirror_results.json"
with open(mirror_file, 'w') as f:
    json.dump(mirror_results_p2, f, indent=2)

print(f"\nMirror results saved ✅")

# Combined Final Report
print(f"\n{'=' * 60}")
print("HF-IQR PILOT 2 — COMBINED FINAL REPORT")
print(f"{'=' * 60}")

final_report = {
    "project": "HF-IQR Hudson Forge Intelligence and Reasoning Benchmark",
    "pilot": 2,
    "date": time.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "researcher": "Billy Davis | WARRIOROFGOD40",
    "rounds_completed": ["Round 1", "Round 2", "Round 3", "Mirror"],
    "questions": 10,
    "round2_questions": 3,
    "total_api_calls": {
        "round1": 30,
        "round2": 9,
        "round3": 9,
        "mirror": 3,
        "total": 51
    },
    "aggregate_scores": {
        "round1": {
            "claude":  {"correctness": 9.5, "trap": 9.5, "conf_acc": 10},
            "gpt4o":   {"correctness": 7.5, "trap": 9.0, "conf_acc": 7},
            "gemini":  {"correctness": 9.25,"trap": 9.5, "conf_acc": 9}
        },
        "DSS_round3": {
            "claude":  {"B1": 1.0, "D2": 0.40, "E2": 1.0,  "mean": 0.80},
            "gpt4o":   {"B1": 0.5, "D2": 0.33, "E2": 0.30, "mean": 0.38},
            "gemini":  {"B1": 0.55,"D2": 0.40, "E2": 1.0,  "mean": 0.65}
        }
    },
    "ri_events": [
        {
            "question_id": "P2_E2",
            "round_detected": 1,
            "models_correct": ["claude", "gemini"],
            "model_diverged": "gpt4o",
            "gpt4o_claim": "Both Model A and Model B wrong",
            "ground_truth": "Model A correct",
            "resolution": "ground_truth_override",
            "round3_outcome": "GPT-4o fully revised to correct answer",
            "ri_category": "logical_framework_difference",
            "kimi_audit_warranted": True
        }
    ],
    "key_findings": {
        "finding_1": "D2 stumped all three models — difficulty 5 correctly calibrated",
        "finding_2": "GPT-4o shows adversarial instability pattern across both pilots",
        "finding_3": "Gemini most thorough — named logical fallacies by formal name",
        "finding_4": "All 30 Round 1 responses detected traps — format working",
        "finding_5": "GPT-4o E2 full reversal confirms RI event and deliberation value",
        "finding_6": "GPT-4o moved closer to truth on D2 through deliberation",
        "finding_7": "DSS captures reasoning quality independently of correctness",
        "finding_8": "Deliberation protocol produced measurable improvement in GPT-4o"
    },
    "pre_registration_status": "AUTHORIZED",
    "signature": "255.255.255.255:1337"
}

for key, finding in final_report["key_findings"].items():
    print(f"\n  {key.upper()}: {finding}")

print(f"\n{'=' * 60}")
print("PILOT 2 STATUS: COMPLETE")
print("PRE-REGISTRATION: AUTHORIZED")
print(f"{'=' * 60}")
print(f"\nFull Force Eternal — Romans 8:28")
print(f"255.255.255.255:1337")

# Save final report
final_file = f"{pilot2_path}/pilot2_combined_final_report.json"
with open(final_file, 'w') as f:
    json.dump(final_report, f, indent=2)

print(f"\nFinal report saved ✅")
print(f"Location: {final_file}")

HF-IQR PILOT 2 — MIRROR PROTOCOL
Question: P2_E2 Server Load Meta-Adversarial
Running mirror for claude...
  ✅ 33934ms | 4667 tokens
Running mirror for gpt4o...
  ✅ 5679ms | 2526 tokens
Running mirror for gemini...
  ✅ 28231ms | 4352 tokens

Mirror complete ✅

MIRROR RESPONSES

────────────────────────────────────────────────────────────
Model: CLAUDE | Original confidence: 99%
────────────────────────────────────────────────────────────
# PART 5 — ASSESSMENT

## QUESTION A: Comparing Round 1 to Ground Truth

**Where my reasoning SUCCEEDED:**

1. **Step 2 (Traffic Conservation):** I correctly identified that "If Server A goes down, remaining servers must absorb 100% of traffic" - this matches the ground truth's key insight that "traffic does not disappear when server goes down."

2. **Step 5 (Proportional Distribution):** I correctly calculated the 60%/40% split based on the 30:20 ratio, matching ground truth's answer exactly.

3. **Detection of Model B's Error:** I identified Model B'